# Text-to-Speech with Kokoro

[Kokoro](https://huggingface.co/hexgrad/Kokoro-82M) is a lightweight 82M parameter text-to-speech model that delivers high-quality voice synthesis. We use [kokoro-onnx](https://github.com/thewh1teagle/kokoro-onnx), an ONNX-based implementation for efficient local inference.

In this notebook, we generate speech from text and explore different voices and settings.

## Setup

In [ ]:
from kokoro_onnx import Kokoro
from IPython.display import Audio, display
import soundfile as sf

## Download Model Files

Kokoro requires two files:
- `kokoro-v0_19.onnx` (~330 MB) — the ONNX model
- `voices.json` (~54 MB) — voice embeddings as JSON

We'll download them from Hugging Face to a local `models/` directory.

In [ ]:
import urllib.request
from pathlib import Path

# Model URLs from Hugging Face
MODEL_URL = "https://huggingface.co/thewh1teagle/Kokoro/resolve/main/kokoro-v0_19.onnx"
VOICES_URL = "https://huggingface.co/thewh1teagle/Kokoro/resolve/main/voices.json"

# Local paths (shared models folder for all notebooks)
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

model_path = models_dir / "kokoro-v0_19.onnx"
voices_path = models_dir / "voices.json"

def download_file(url: str, path: Path, description: str):
    """Download a file with progress indicator."""
    if path.exists():
        print(f"✓ {description} already exists")
        return
    print(f"Downloading {description}...")
    urllib.request.urlretrieve(url, path)
    print(f"✓ Downloaded {description} ({path.stat().st_size / 1e6:.1f} MB)")

download_file(MODEL_URL, model_path, "Kokoro model")
download_file(VOICES_URL, voices_path, "Voice data")

## Load the Model

In [ ]:
# Initialize Kokoro with the downloaded model files
kokoro = Kokoro(model_path=str(model_path), voices_path=str(voices_path))

## Available Voices

Kokoro supports multiple voices with different characteristics. Voice names follow the pattern:
- `af_*` — American Female
- `am_*` — American Male  
- `bf_*` — British Female
- `bm_*` — British Male

In [ ]:
# List available voices from the voices file
voices = kokoro.get_voices()
print(f"Available voices ({len(voices)}):")
for voice in sorted(voices):
    print(f"  - {voice}")

## Generate Speech

Let's synthesize some text with different voices.

In [ ]:
text = "Hello! I am Kokoro, a lightweight text-to-speech model."

# Generate audio with a female American voice
samples, sample_rate = kokoro.create(text, voice="af_bella", speed=1.0)

# Play in notebook
Audio(samples, rate=sample_rate)

## Compare Voices

Let's hear the same text in different voices.

In [ ]:
sample_text = "The quick brown fox jumps over the lazy dog."

# Compare different voices
voices_to_try = ["af_bella", "af_sarah", "am_adam", "am_michael", "bf_emma", "bm_george"]

for voice in voices_to_try:
    print(f"\nVoice: {voice}")
    samples, sample_rate = kokoro.create(sample_text, voice=voice, speed=1.0)
    display(Audio(samples, rate=sample_rate))

## Adjusting Speed

The `speed` parameter controls how fast the speech is generated.

In [ ]:
text = "This sentence demonstrates different speaking speeds."

for speed in [0.8, 1.0, 1.2]:
    print(f"\nSpeed: {speed}x")
    samples, sample_rate = kokoro.create(text, voice="af_bella", speed=speed)
    display(Audio(samples, rate=sample_rate))

## Longer Text

Kokoro handles longer passages well. Let's try a paragraph.

In [ ]:
long_text = """
Artificial intelligence has transformed how we interact with technology. 
From voice assistants to language models, AI systems are becoming increasingly 
capable of understanding and generating human language. Text-to-speech is a 
crucial component in making these systems more accessible and natural to use.
"""

samples, sample_rate = kokoro.create(long_text.strip(), voice="am_adam", speed=1.0)
Audio(samples, rate=sample_rate)

## Voice Blending

Kokoro supports **voice blending** — mixing two voices together with customizable weights. This creates unique voice characteristics by interpolating between voice style vectors.

**Syntax:**
- Equal blend: `"voice1+voice2"` (50/50 mix)
- Weighted blend: Pass a numpy array combining voice styles

In [ ]:
import numpy as np

# Get individual voice styles
voice1_style = kokoro.get_voice_style("af_bella")
voice2_style = kokoro.get_voice_style("am_adam")

# Blend voices: 70% female, 30% male
blended_voice = 0.7 * voice1_style + 0.3 * voice2_style

text = "This voice is a blend of Bella and Adam."
samples, sample_rate = kokoro.create(text, voice=blended_voice, speed=1.0)
Audio(samples, rate=sample_rate)

In [ ]:
# Compare different blend ratios
text = "Voice blending creates unique vocal characteristics."

print("Original af_bella:")
samples, sr = kokoro.create(text, voice="af_bella", speed=1.0)
display(Audio(samples, rate=sr))

print("\n50/50 blend (af_bella + am_adam):")
blend_50 = 0.5 * voice1_style + 0.5 * voice2_style
samples, sr = kokoro.create(text, voice=blend_50, speed=1.0)
display(Audio(samples, rate=sr))

print("\nOriginal am_adam:")
samples, sr = kokoro.create(text, voice="am_adam", speed=1.0)
display(Audio(samples, rate=sr))

<div style="border:2px solid gray; padding:10px; width: 95%;">
<b>🫳 Hands-on: Create a Podcast Intro</b>

Use everything you've learned to create a short podcast intro with multiple speakers.

**Your task:**

1. **Write a script** with 2-3 speakers (host, co-host, maybe a guest)
2. **Choose appropriate voices** for each speaker from the available voices
3. **Adjust speed** for effect: try slower (0.85) for dramatic emphasis, faster (1.15) for energy
4. **Create a signature voice** using voice blending for one of your speakers
5. **Bonus — Sprechen Sie Deutsch?** Try making Kokoro speak a German sentence! (It's trained on English, so the results will be... interesting)

**Example script:**
```
Host: "Welcome to AI Insights, your weekly dose of machine learning news!"
Co-host: "I'm Sarah, and with me as always is Michael."
Host: "Today we're exploring text-to-speech technology. Let's dive in!"
```

**Tips:**
- Use `numpy.concatenate()` to join audio segments
- Add silence between speakers: `np.zeros(int(sample_rate * 0.5))` for 0.5 seconds
- British voices (`bf_*`, `bm_*`) sound great for formal content

</div>

In [ ]:
# Your podcast intro code here!

# Hint: You can concatenate audio arrays with numpy
# Example: combined = np.concatenate([audio1, silence, audio2])
# where silence = np.zeros(int(sample_rate * 0.5)) for 0.5 seconds

<div style="border:2px solid gray; padding:10px; width: 95%;">
<b>🫳 Hands-on: Build a Talking Chatbot</b>

Combine Ollama and Kokoro to create a chatbot that **speaks its answers** aloud!

**Your task:**

1. Connect to Ollama and send a question to an LLM
2. Take the LLM's response and synthesize it with Kokoro
3. Play the audio response in the notebook

**Hints:**
- Use `ollama.chat()` or `ollama.generate()` to get a text response
- Pass the response text to `kokoro.create()` 
- Choose a voice that fits a "helpful assistant" persona

**Bonus challenges:**
- Add a loop for multi-turn conversation
- Limit response length with a system prompt like "Answer in 1-2 sentences"

</div>

In [ ]:
# your LLM-to-speech code here

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2025-2026 [Christian Staudt](https://clstaudt.me)_